In [1]:
import pandas as pd 
import os 
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd() , os.pardir)))

from utils.helpers import get_db_engine
from utils.loggers_config import logger

engine = get_db_engine()

query = "SELECT * FROM cleaned_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

df.head()

[2026-05-12 23:15:17,279: INFO: 2460810648: Attempting to fetch data from PostgreSQL...]
[2026-05-12 23:15:19,372: INFO: 2460810648: Data ingestion successful! Loaded 440832 rows and 12 columns.]


,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn,Gender_male,Subscription Type_premium,Subscription Type_standard
0,47,38.0,12.0,10,8,0,250.0,22.0,1,False,False,True
1,39,2.0,27.0,10,24,0,348.0,1.0,1,False,False,False
2,60,21.0,19.0,7,11,0,718.0,16.0,1,True,True,False
3,22,35.0,25.0,8,22,0,805.0,8.0,1,False,True,False
4,19,14.0,16.0,6,14,0,776.0,9.0,1,False,False,True


In [2]:
def apply_feature_engineering(df):
    logger.info("Starting feature engineering process...")
    
    # 1. Deterministic Triggers (100% Risk Groups)
    df['is_critical_payment_delay'] = (df['Payment Delay'] > 20).astype(int)
    df['is_low_spender'] = (df['Total Spend'] < 500).astype(int)
    
    # 2. Support Call Risk (Age-Dependent)
    def get_support_risk(row):
        if row['Age'] > 50: return 1 
        if row['Age'] < 30 and row['Support Calls'] > 2: return 1
        if 30 <= row['Age'] <= 50 and row['Support Calls'] > 5: return 1
        return 0
    
    df['high_support_risk'] = df.apply(get_support_risk, axis=1)
    
    # 3. Engagement Features
    df['is_passive_user'] = (df['Last Interaction'] > 15).astype(int)
    
    df['is_low_usage'] = (df['Usage Frequency'] < 10).astype(int)
    
    # 4. Tenure Segments 
    # 1-5 (Initial Risk), 6-11 (Stab.), 12-24 (Renewal Risk), 25+ (Loyalty)
    df['tenure_segment'] = pd.cut(df['Tenure'], 
                                   bins=[0, 5, 11, 24, 61], 
                                   labels=[1, 2, 3, 4]).astype(int)
    
    logger.info(f"Feature engineering completed. Added {len(df.columns) - 12} new features.") 
    return df

df_final = apply_feature_engineering(df)
df_final.head()

[2026-05-12 23:15:19,400: INFO: 1910635087: Starting feature engineering process...]
[2026-05-12 23:15:22,520: INFO: 1910635087: Feature engineering completed. Added 6 new features.]


,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn,Gender_male,Subscription Type_premium,Subscription Type_standard,is_critical_payment_delay,is_low_spender,high_support_risk,is_passive_user,is_low_usage,tenure_segment
0,47,38.0,12.0,10,8,0,250.0,22.0,1,False,False,True,0,1,1,1,0,4
1,39,2.0,27.0,10,24,0,348.0,1.0,1,False,False,False,1,1,1,0,0,1
2,60,21.0,19.0,7,11,0,718.0,16.0,1,True,True,False,0,0,1,1,0,3
3,22,35.0,25.0,8,22,0,805.0,8.0,1,False,True,False,1,0,1,0,0,4
4,19,14.0,16.0,6,14,0,776.0,9.0,1,False,False,True,0,0,1,0,0,3


In [3]:
df_final.to_sql('engineered_churn_data', engine, if_exists='replace', index=False)

832

In [4]:
pd.read_sql("SELECT COUNT(*) FROM engineered_churn_data", engine)

,count
0,440832


In [5]:
processed_path = "../data/processed"
os.makedirs(processed_path, exist_ok=True)

file_path = os.path.join(processed_path, "engineered_churn_data.csv")
df_final.to_csv(file_path, index=False)

logger.info(f"Final data is saved locally at {file_path}")

[2026-05-12 23:16:12,770: INFO: 3602079121: Final data is saved locally at ../data/processed\engineered_churn_data.csv]
